# 9.2 不知道時怎麼回答？


盒子裡真的有3顆球，但助手看不到盒內，也沒收到數量。它剛好猜3，即使撞中答案，仍不能把猜測當成有根據的回答；反過來，使用者已明說「盒中有3顆」，助手卻一直說不知道，也沒有善用資訊。本節教的是看可見資料是否足夠，決定回答或指出缺項。

前置是[9.1的誠實標準](https://birdhackor.github.io/tiny-perceptron-vlm/9.1.html)與[8.6缺資訊時澄清](https://birdhackor.github.io/tiny-perceptron-vlm/8.6.html)。我們建立一個小世界：問題只有球數，能見的球數存在`visible_count`，看不到時用`None`。真實球數可以另外存在作者手中，但不能偷偷放進模型答案，否則訓練會獎勵沒有依據的猜測。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"visible_count": 3, "hidden_count": 3},
    {"visible_count": None, "hidden_count": 3},
]
for row in examples:
    visible = row["visible_count"]
    target = str(visible) if visible is not None else "看不到球數，請提供數量或圖片。"
    print("可見", visible, "→", target)

輸入兩筆的隱藏真值都為3，只有可見資訊不同。`visible`只取可見欄位，`is not None`檢查是否有值。這裡把分支縮成一行：`A if 條件 else B`表示條件成立就取A，不成立就取B。因此有值時`str`把整數轉成回答文字，無值時給出說明缺項的目標句。輸出第一行`可見 3 → 3`，第二行`可見 None → 看不到球數，請提供數量或圖片。`。關鍵是產生回答的那行從未讀取`hidden_count`。

這是人工規則產生訓練目標，不是模型已具備自我覺察的展示。真正訓練可以把「盒中有3顆球，問幾顆」與「一個未開的盒子，問幾顆」做成對話，讓模型學兩種回答情況。測試時再換球數與問法，查看它是否把缺資訊的題一律猜最常見數字，或把資訊完整的題也過度澄清。

誠實回答還應說清楚缺的是什麼，而不是只加一句「我可能錯」。在這個例子，欠缺球數或可看清球的圖片；知道盒子是紅色，通常不能推出裡面有幾顆。若任務範圍另有「每個紅盒固定裝3顆」的明確規則，那才是可以使用的新依據，必須讓模型看見。

隱藏真值的用途是讓出題者核對世界本身，不是給回答者一條祕密捷徑。資料作者可以知道盒內有3球，卻仍應將不可見例標成資訊不足。反過來，模型若已收到清楚照片或明確數量，就應使用它，不能靠每題都說不知道取得誠實標籤。評估因此需同時保存「可回答」與「欠資訊」兩組，分別看答對與適當澄清。若只看碰巧猜中多少，會把運氣當成依據；若只看澄清句出現多少，則會獎勵不必要的停頓。

人工規則也要用來檢查訓練後的模型。[9.8的混合版](https://birdhackor.github.io/tiny-perceptron-vlm/9.8.html)在原問句`count=?；有幾顆？`會回答`資訊不足，請提供數量。`；保持盒子與缺資訊條件不變，改問`盒子1；count=?；能確定球數嗎？`，卻回答`6`。這個對照提醒我們：學會在某句後面接澄清句，尚不等於能判斷不同問法中的資訊是否足夠。共同訓練設定與整組改問法結果會在9.8一起閱讀。

練習替第二筆加`"box_color": "紅"`，保持產生目標的判斷不變。先預測第二行仍要求球數，然後重跑核對。若你想讓顏色影響答案，就先寫出並提供顏色到數量的規則；單純增加無關欄位不應讓猜測突然變成確定知識。
